In [ ]:
import pandas as pd

file=r"J:\ctgroup\Edward\DATA\VMI\20250408\calibrated\po_500mW_ell_calibrated.h5"

data = pd.read_hdf(file, key='data')

In [ ]:
import plotly.express as px

data=data[data['m/q']<65]

px.histogram(data, x='m/q',nbins=512).show()

gates={
    'CH4': (12,16),
    'C2H4': (23,32),
    'C2H3O': (37,47),
    'Parent': (52,59),
}

def gate_data(row):
    for gate, (low, high) in gates.items():
        if low<=row['m/q']<=high:
            return gate
    return 'Other'

data['gate']=data.apply(gate_data, axis=1)

In [ ]:
px.histogram(data,x='m/q', color='gate', nbins=512).show()

In [ ]:
import numpy as np
for gate in gates.keys():
    subset=data[data['gate']==gate]
    subset=subset[subset['pz'].between(-0.5, 0.5)]
    px.density_heatmap(
            x=subset['px'], y=subset['pz'], color_continuous_scale='Viridis', nbinsx=512, nbinsy=512, title=gate
    ).show()
    dir=r"J:\ctgroup\Edward\DATA\VMI\20250408\calibrated\gates"
    subset.to_hdf(fr"{dir}\{gate}_gate.h5", key='data', mode='w')
    print(fr"Saved {gate} gate data to {dir}\{gate}_gate.h5")

    rhist_full, re_full=np.histogram(np.sqrt(subset['px']**2+subset['py']**2+subset['pz']**2), bins=512, range=[0, 0.5])

    # np.save(fr"{dir}\{gate}_full_rhist.npy", rhist_full)

    subset=subset[np.abs(subset['py'])<0.05]
    px.density_heatmap(
            x=subset['px'], y=subset['pz'], color_continuous_scale='Viridis', nbinsx=512, nbinsy=512, title=f"{gate} (|pz|<0.05)"
    ).show()

    hist, xe, ye=np.histogram2d(subset['px'], subset['py'], bins=512, range=[[-0.5, 0.5], [-0.5, 0.5]])
    xe, ye = (xe[:-1] + xe[1:]) / 2, (ye[:-1] + ye[1:]) / 2
    # np.save(fr"{dir}\{gate}_slice_hist.npy", hist)
    # np.save(fr"{dir}\x.npy", xe)
    # np.save(fr"{dir}\y.npy", ye)

    rhist, re=np.histogram(np.sqrt(subset['px']**2+subset['py']**2), bins=512, range=[0, 0.5])
    re = (re[:-1] + re[1:]) / 2

    # np.save(fr"{dir}\{gate}_slice_rhist.npy", rhist)
    # np.save(fr"{dir}\r.npy", re)

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
for gate in gates.keys():
    subset=data[data['gate']==gate]
    n_zero_energy_2d=np.sum((subset['px']**2+subset['py']**2)<0.015**2)

    slice=subset[np.abs(subset['pz'])<0.1]
    n_zero_energy_3d=np.sum((slice['px']**2+slice['py']**2)<0.015**2)
    n_total=len(subset)
    n_slice=len(slice)
    print(f"3D: {gate}: {n_zero_energy_3d} / {n_total} ({n_zero_energy_3d/n_total:.2%})")
    print(f"2D: {gate}: {n_zero_energy_2d} / {n_total} ({n_zero_energy_2d/n_total:.2%})")

    print(f"diff: {gate}: {n_zero_energy_2d/n_total - n_zero_energy_3d/n_total:.2%}")

    fig= make_subplots(rows=2, cols=1, subplot_titles=[f"Projection: {gate}", f"Slice: {gate}"], vertical_spacing=0.05)

    hist, xe, ye=np.histogram2d(subset['px'], subset['py'], bins=512, range=[[-0.5, 0.5], [-0.5, 0.5]])
    hist=hist.T
    fig.add_trace(go.Heatmap(x=xe, y=ye, z=slice_hist, showscale=False), row=1, col=1)

    hist, xe, ye=np.histogram2d(slice['px'], slice['py'], bins=512, range=[[-0.5, 0.5], [-0.5, 0.5]])
    hist=hist.T
    fig.add_trace(go.Heatmap(x=xe, y=ye, z=hist, showscale=False), row=2, col=1)

    circle_theta=np.linspace(0, 2*np.pi, 100)

    cx=0.015*np.cos(circle_theta)
    cy=0.015*np.sin(circle_theta)

    fig.add_scatter(x=cx, y=cy, mode='lines', line=dict(color='red', width=2),  row=1, col=1)
    fig.add_annotation(
            x= 0.02, y= 0.02, text=f"Zero Energy: {n_zero_energy_2d/n_total:.2%}", showarrow=True, row=1, col=1, font=dict(size=16, color='white')
    )
    fig.add_scatter(x=cx, y=cy, mode='lines', line=dict(color='red', width=2), row=2, col=1)
    fig.add_annotation(
            x= 0.02, y= 0.02, text=f"Zero Energy: {n_zero_energy_3d/n_total:.2%}", showarrow=True, row=2, col=1, font=dict(size=16, color='white')
    )

    fig.update_layout(height=1000, width=600, showlegend=False)
    fig.write_image(
            fr"J:\ctgroup\Edward\DATA\VMI\20250408\calibrated\{gate}.png", scale=2
    )
    fig.show()

    # print(f"3D vs 2D: {gate}: {n_zero_energy_3d} / {n_zero_energy_2d} ({n_zero_energy_3d/n_zero_energy_2d:.2%})")
